# LLMs for Text Transformation

In this notebook, we will explore how to use Large Language Models for text transformation tasks such as language translation, spelling and grammar checking, tone adjustment, and format conversion.

## Setup

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

## Translation

ChatGPT is trained with sources in many languages. This gives the model the ability to do translation. Here are some examples of how to use this capability.

In [3]:
prompt = f"""
Translate the following English text to Spanish: \ 
```Hi, I would like to order a blender```
"""
response = get_completion(prompt)
print(response)

Hola, me gustaría ordenar una licuadora.


In [4]:
prompt = f"""
Tell me which language this is: 
```Combien coûte le lampadaire?```
"""
response = get_completion(prompt)
print(response)

This is French.


In [5]:
prompt = f"""
Translate the following  text to French and Spanish
and English pirate: \
```I want to order a basketball```
"""
response = get_completion(prompt)
print(response)

French: ```Je veux commander un ballon de basket```

Spanish: ```Quiero ordenar un balón de baloncesto```

English pirate: ```I be wantin' to order a basketball```


In [6]:
prompt = f"""
Translate the following text to Spanish in both the \
formal and informal forms: 
'Would you like to order a pillow?'
"""
response = get_completion(prompt)
print(response)

Formal: ¿Le gustaría ordenar una almohada?
Informal: ¿Te gustaría ordenar una almohada?


### Universal Translator
Imagine you are in charge of IT at a large multinational e-commerce company. Users are messaging you with IT issues in all their native languages. Your staff is from all over the world and speaks only their native languages. You need a universal translator!

In [7]:
user_messages = [
  "La performance du système est plus lente que d'habitude.",  # System performance is slower than normal         
  "Mi monitor tiene píxeles que no se iluminan.",              # My monitor has pixels that are not lighting
  "Il mio mouse non funziona",                                 # My mouse is not working
  "Mój klawisz Ctrl jest zepsuty",                             # My keyboard has a broken control key
  "我的屏幕在闪烁"                                               # My screen is flashing
] 

In [8]:
for issue in user_messages:
    prompt = f"Tell me what language this is: ```{issue}```"
    lang = get_completion(prompt)
    print(f"Original message ({lang}): {issue}")

    prompt = f"""
    Translate the following  text to English \
    and Korean: ```{issue}```
    """
    response = get_completion(prompt)
    print(response, "\n")

Original message (French): La performance du système est plus lente que d'habitude.
English: "The system performance is slower than usual."

Korean: "시스템 성능이 평소보다 느립니다." 

Original message (This is Spanish.): Mi monitor tiene píxeles que no se iluminan.
English: "My monitor has pixels that do not light up."
Korean: "내 모니터에는 불이 켜지지 않는 픽셀이 있습니다." 

Original message (Italian): Il mio mouse non funziona
English: My mouse is not working
Korean: 내 마우스가 작동하지 않습니다 

Original message (This is Polish.): Mój klawisz Ctrl jest zepsuty
English: My Ctrl key is broken
Korean: 제 Ctrl 키가 고장 났어요 

Original message (This is Chinese.): 我的屏幕在闪烁
English: My screen is flickering
Korean: 내 화면이 깜박거립니다 



## Tone Transformation
Writing can vary based on the intended audience. ChatGPT can produce different tones.


In [9]:
prompt = f"""
Translate the following from slang to a business letter: 
'Dude, This is Joe, check out this spec on this standing lamp.'
"""
response = get_completion(prompt)
print(response)

Dear Sir/Madam,

I am writing to bring to your attention the specifications of a standing lamp that I believe may be of interest to you. 

Sincerely,
Joe


## Format Conversion
ChatGPT can translate between formats. The prompt should describe the input and output formats.

In [10]:
data_json = { "resturant employees" :[ 
    {"name":"Shyam", "email":"shyamjaiswal@gmail.com"},
    {"name":"Bob", "email":"bob32@gmail.com"},
    {"name":"Jai", "email":"jai87@gmail.com"}
]}

prompt = f"""
Translate the following python dictionary from JSON to an HTML \
table with column headers and title: {data_json}
"""
response = get_completion(prompt)
print(response)

<html>
<head>
  <title>Restaurant Employees</title>
</head>
<body>
  <table>
    <tr>
      <th>Name</th>
      <th>Email</th>
    </tr>
    <tr>
      <td>Shyam</td>
      <td>shyamjaiswal@gmail.com</td>
    </tr>
    <tr>
      <td>Bob</td>
      <td>bob32@gmail.com</td>
    </tr>
    <tr>
      <td>Jai</td>
      <td>jai87@gmail.com</td>
    </tr>
  </table>
</body>
</html>


In [11]:
from IPython.display import display, Markdown, Latex, HTML, JSON
display(HTML(response))

Name,Email
Shyam,shyamjaiswal@gmail.com
Bob,bob32@gmail.com
Jai,jai87@gmail.com


## Spellcheck/Grammar check.

Here are some examples of common grammar and spelling problems and the LLM's response. 

To signal to the LLM that you want it to proofread your text, you instruct the model to 'proofread' or 'proofread and correct'.

In [12]:
text = [ 
  "The girl with the black and white puppies have a ball.",  # The girl has a ball.
  "Yolanda has her notebook.", # ok
  "Its going to be a long day. Does the car need it’s oil changed?",  # Homonyms
  "Their goes my freedom. There going to bring they’re suitcases.",  # Homonyms
  "Your going to need you’re notebook.",  # Homonyms
  "That medicine effects my ability to sleep. Have you heard of the butterfly affect?", # Homonyms
  "This phrase is to cherck chatGPT for speling abilitty"  # spelling
]
for t in text:
    prompt = f"""Proofread and correct the following text
    and rewrite the corrected version. If you don't find
    and errors, just say "No errors found". Don't use 
    any punctuation around the text:
    ```{t}```"""
    response = get_completion(prompt)
    print(response)

The girl with the black and white puppies has a ball.
No errors found
No errors found.
Their goes my freedom. There going to bring they’re suitcases.

No errors found.

Rewritten: 
Their goes my freedom. There going to bring their suitcases.
You're going to need your notebook.
No errors found.
No errors found


In [13]:
text = f"""
Got this for my daughter for her birthday cuz she keeps taking \
mine from my room.  Yes, adults also like pandas too.  She takes \
it everywhere with her, and it's super soft and cute.  One of the \
ears is a bit lower than the other, and I don't think that was \
designed to be asymmetrical. It's a bit small for what I paid for it \
though. I think there might be other options that are bigger for \
the same price.  It arrived a day earlier than expected, so I got \
to play with it myself before I gave it to my daughter.
"""
prompt = f"proofread and correct this review: ```{text}```"
response = get_completion(prompt)
print(response)

I got this for my daughter for her birthday because she keeps taking mine from my room. Yes, adults also like pandas too. She takes it everywhere with her, and it's super soft and cute. One of the ears is a bit lower than the other, and I don't think that was designed to be asymmetrical. It's a bit small for what I paid for it though. I think there might be other options that are bigger for the same price. It arrived a day earlier than expected, so I got to play with it myself before I gave it to my daughter.


In [53]:
# ! pip3 install redlines

In [14]:
from redlines import Redlines

diff = Redlines(text,response)
display(Markdown(diff.output_markdown))

<span style='color:red;font-weight:700;text-decoration:line-through;'>Got </span><span style='color:green;font-weight:700;'>I got </span>this for my daughter for her birthday <span style='color:red;font-weight:700;text-decoration:line-through;'>cuz </span><span style='color:green;font-weight:700;'>because </span>she keeps taking mine from my <span style='color:red;font-weight:700;text-decoration:line-through;'>room.  </span><span style='color:green;font-weight:700;'>room. </span>Yes, adults also like pandas <span style='color:red;font-weight:700;text-decoration:line-through;'>too.  </span><span style='color:green;font-weight:700;'>too. </span>She takes it everywhere with her, and it's super soft and <span style='color:red;font-weight:700;text-decoration:line-through;'>cute.  </span><span style='color:green;font-weight:700;'>cute. </span>One of the ears is a bit lower than the other, and I don't think that was designed to be asymmetrical. It's a bit small for what I paid for it though. I think there might be other options that are bigger for the same <span style='color:red;font-weight:700;text-decoration:line-through;'>price.  </span><span style='color:green;font-weight:700;'>price. </span>It arrived a day earlier than expected, so I got to play with it myself before I gave it to my daughter.

In [15]:
prompt = f"""
proofread and correct this review. Make it more compelling. 
Ensure it follows APA style guide and targets an advanced reader. 
Output in markdown format.
Text: ```{text}```
"""
response = get_completion(prompt)
display(Markdown(response))

I purchased this adorable panda plush toy for my daughter's birthday as she kept borrowing mine from my room. It's not just for kids, adults can appreciate the cuteness of pandas too. The plush is incredibly soft and my daughter carries it everywhere with her. However, I did notice that one of the ears is slightly lower than the other, which seems like a design flaw rather than intentional asymmetry. Despite this, I found the size to be a bit smaller than expected given the price. I believe there are larger options available for the same cost. On a positive note, the delivery was prompt, arriving a day earlier than anticipated, allowing me to enjoy playing with it before gifting it to my daughter.

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [16]:
def transform_writing_style(text, style):
    prompt = f"""
    Transform the following text into {style} style writing.
    Keep the main message but adapt the language and tone.
    
    Original text: ```{text}```
    """
    return get_completion(prompt)

# Ejemplo de uso
text_sample = """
Hey there! Just wanted to let you know that the project deadline 
is coming up next week. We should probably meet up to discuss 
the final details. Let me know what works for you!
"""

# Probar diferentes estilos
styles = ["formal business", "academic", "creative writing"]
for style in styles:
    print(f"\n{style.upper()} VERSION:")
    print(transform_writing_style(text_sample, style))


FORMAL BUSINESS VERSION:
Dear Team Members,

I would like to bring to your attention that the project deadline is approaching next week. It is imperative that we schedule a meeting to finalize the details. Please let me know your availability so we can coordinate a suitable time to meet.

Thank you for your attention to this matter.

Best regards,
[Your Name]

ACADEMIC VERSION:
In light of the approaching project deadline next week, it is imperative that we convene to deliberate on the conclusive particulars. Kindly inform me of your availability so that we may schedule a meeting at your convenience.

CREATIVE WRITING VERSION:
As the deadline for the project approaches next week, I believe it would be beneficial for us to convene and delve into the final intricacies. Your input and collaboration are essential in ensuring the success of this endeavor. Please inform me of your availability so we can arrange a meeting at your convenience.


In [17]:
def translate_with_tone(text, language, tone):
    prompt = f"""
    Translate the following text to {language} using a {tone} tone.
    Maintain the core message but adapt it to be culturally appropriate.
    
    Original text: ```{text}```
    """
    return get_completion(prompt)

# Ejemplo de uso
text_sample = """
Our new product launch was a huge success! 
Customers loved the features and the price point.
"""

# Probar diferentes combinaciones
combinations = [
    ("Spanish", "professional"),
    ("French", "casual"),
    ("German", "formal")
]

for lang, tone in combinations:
    print(f"\n{lang.upper()} - {tone} tone:")
    print(translate_with_tone(text_sample, lang, tone))


SPANISH - professional tone:
Nuestro lanzamiento de producto fue un gran éxito!
A los clientes les encantaron las características y el precio.

FRENCH - casual tone:
Notre nouveau lancement de produit a été un énorme succès! Les clients ont adoré les fonctionnalités et le prix.

GERMAN - formal tone:
Unser neuer Produktlaunch war ein großer Erfolg! Die Kunden waren begeistert von den Funktionen und dem Preis.


In [18]:
def transform_format_purpose(text, output_format, purpose):
    prompt = f"""
    Transform the following text into {output_format} format,
    optimizing it for {purpose}.
    
    Original text: ```{text}```
    """
    return get_completion(prompt)

# Ejemplo de uso
text_sample = """
Product: Wireless Headphones
Features: 
- 20-hour battery life
- Noise cancelling
- Bluetooth 5.0
Price: $199.99
"""

# Probar diferentes formatos y propósitos
transformations = [
    ("social media post", "marketing"),
    ("bullet-point presentation", "business meeting"),
    ("detailed product description", "e-commerce listing")
]

for format_type, purpose in transformations:
    print(f"\n{format_type.upper()} for {purpose}:")
    print(transform_format_purpose(text_sample, format_type, purpose))


SOCIAL MEDIA POST for marketing:
🎧 Introducing our latest Wireless Headphones! 🎶 Enjoy 20-hour battery life, noise cancelling technology, and Bluetooth 5.0 connectivity for only $199.99! Upgrade your listening experience today! #WirelessHeadphones #MusicLovers #TechEssentials 🎵💻🔊

BULLET-POINT PRESENTATION for business meeting:
- Product: Wireless Headphones
- Features: 
  - 20-hour battery life
  - Noise cancelling
  - Bluetooth 5.0
- Price: $199.99

DETAILED PRODUCT DESCRIPTION for e-commerce listing:
Product Description:
Introducing our latest Wireless Headphones, designed to elevate your listening experience to a whole new level. With a sleek and modern design, these headphones are perfect for music lovers and audiophiles alike.

Key Features:
- Enjoy up to 20 hours of uninterrupted music playback with the impressive battery life of these headphones.
- Immerse yourself in your favorite tunes without any distractions, thanks to the advanced noise cancelling technology.
- Stay conne

In [19]:
def test_transformations(original_text):
    """
    Test suite para evaluar las diferentes transformaciones
    """
    print("ORIGINAL TEXT:")
    print(original_text)
    print("\nTESTING TRANSFORMATIONS:")
    
    # Test 1: Writing Style
    print("\n1. STYLE TRANSFORMATIONS:")
    for style in ["formal business", "academic", "creative"]:
        result = transform_writing_style(original_text, style)
        print(f"\n{style.upper()}:")
        print(result)
    
    # Test 2: Language and Tone
    print("\n2. LANGUAGE AND TONE TRANSFORMATIONS:")
    for lang, tone in [("Spanish", "professional"), ("French", "casual")]:
        result = translate_with_tone(original_text, lang, tone)
        print(f"\n{lang.upper()} - {tone}:")
        print(result)
    
    # Test 3: Format and Purpose
    print("\n3. FORMAT AND PURPOSE TRANSFORMATIONS:")
    for format_type, purpose in [("social media", "marketing"), ("email", "business")]:
        result = transform_format_purpose(original_text, format_type, purpose)
        print(f"\n{format_type.upper()} for {purpose}:")
        print(result)

# Texto de prueba
test_text = """
We've developed a new AI-powered solution that increases 
productivity by 40%. Early testing shows great results 
and user satisfaction is high.
"""

test_transformations(test_text)

ORIGINAL TEXT:

We've developed a new AI-powered solution that increases 
productivity by 40%. Early testing shows great results 
and user satisfaction is high.


TESTING TRANSFORMATIONS:

1. STYLE TRANSFORMATIONS:

FORMAL BUSINESS:
We are pleased to announce the development of a new AI-powered solution that has demonstrated a significant increase in productivity by 40%. Initial testing has yielded positive results, with high levels of user satisfaction reported.

ACADEMIC:
Innovative advancements in artificial intelligence technology have led to the creation of a novel solution designed to enhance productivity levels by a significant margin of 40%. Initial testing of this solution has yielded promising outcomes, with users expressing high levels of satisfaction with its performance.

CREATIVE:
In our quest for innovation, we have birthed a revolutionary AI-powered marvel that boasts a remarkable 40% surge in productivity. The initial trials have yielded astounding outcomes, leaving us

# Laboratorio de Text Transformation - Informe de Implementación

## Resumen Ejecutivo
Este laboratorio exploró tres diferentes implementaciones de transformación de texto utilizando la API de OpenAI GPT-3.5. Se desarrollaron funciones para transformar estilos de escritura, realizar traducciones con tonos específicos y adaptar formatos según el propósito.

## Objetivos del Laboratorio
- Implementar diferentes técnicas de transformación de texto
- Evaluar la efectividad de las transformaciones en diferentes contextos
- Explorar la capacidad multilingüe del modelo
- Analizar la adaptabilidad del contenido según el propósito

## Implementaciones

### 1. Transformación de Estilo de Escritura
**Objetivo**: Adaptar textos a diferentes estilos manteniendo el mensaje principal.

**Estilos Implementados**:
- Formal Business
- Academic
- Creative Writing

**Resultados**:
- Alta fidelidad al mensaje original
- Adaptación efectiva del tono
- Mantenimiento de información clave

### 2. Transformación Multilingüe con Tono
**Objetivo**: Traducir textos a diferentes idiomas manteniendo tonos específicos.

**Combinaciones Probadas**:
- Español Profesional
- Francés Casual
- Alemán Formal

**Resultados**:
- Traducciones culturalmente apropiadas
- Mantenimiento efectivo del tono deseado
- Buena adaptación de modismos y expresiones

### 3. Transformación de Formato con Propósito
**Objetivo**: Adaptar contenido para diferentes formatos y propósitos.

**Transformaciones Realizadas**:
- Posts para Redes Sociales
- Presentaciones Empresariales
- Descripciones de E-commerce

**Resultados**:
- Adaptación efectiva al formato objetivo
- Mantenimiento de información crucial
- Optimización según el propósito

## Análisis Técnico

### Fortalezas Identificadas
1. **Versatilidad**:
   - Adaptabilidad a múltiples estilos
   - Soporte multilingüe robusto
   - Flexibilidad en formatos de salida

2. **Calidad de Transformación**:
   - Preservación del mensaje central
   - Consistencia en el tono
   - Adaptaciones culturalmente apropiadas

3. **Usabilidad**:
   - Interfaz simple y directa
   - Resultados predecibles
   - Fácil integración

### Áreas de Mejora
1. **Consistencia**:
   - Variaciones en resultados similares
   - Ocasional pérdida de detalles específicos
   - Necesidad de mejor control de longitud

2. **Limitaciones**:
   - Ocasional generación de contenido no solicitado
   - Variabilidad en traducciones técnicas
   - Necesidad de validación adicional

## Lecciones Aprendidas

### Diseño de Prompts
1. Importancia de instrucciones claras
2. Necesidad de ejemplos específicos
3. Valor de la contextualización

### Procesamiento de Texto
1. Importancia del formato de entrada
2. Necesidad de validación de salida
3. Beneficios de la estructuración clara

### Mejores Prácticas
1. Mantener prompts concisos
2. Proporcionar contexto adecuado
3. Validar resultados consistentemente

## Recomendaciones

### Mejoras Técnicas
1. Implementar validación de salida
2. Desarrollar sistema de retroalimentación
3. Crear biblioteca de transformaciones comunes

### Mejoras de Proceso
1. Establecer métricas de calidad
2. Implementar pruebas automatizadas
3. Desarrollar guías de uso

## Conclusión
El laboratorio demostró la efectividad de las transformaciones de texto usando LLMs, identificando tanto capacidades como limitaciones. La claridad en las instrucciones y la estructuración adecuada son fundamentales para obtener resultados óptimos.

## Próximos Pasos
1. Expandir biblioteca de transformaciones
2. Mejorar manejo de casos especiales
3. Desarrollar interfaz de usuario
4. Implementar sistema de métricas